# DT4 agent: the forest biomass twin over MCP

DT4 serves the forest above-ground biomass (AGB) and carbon twin as an MCP server, `dt4_server.py`.
It observes woodland with **ESA Biomass CCI v7** (100 m, 2005-2012 and 2015-2024; the only data
it is calibrated against), grows stands with two interchangeable growth models, **Chapman-Richards
yield curves** indexed by yield class and the **3-PG** process model, and applies management and
disturbance scenarios with full carbon bookkeeping (living biomass, deadwood, litter, harvested
wood products). Pilot landscapes: **Thetford** (Corsican pine) and **Kielder** (Sitka spruce).

This notebook acts as the orchestrator: it launches the server over stdio, lists its tools and
calls each of them, exactly as an orchestrating agent would.

In [1]:
import sys
from pathlib import Path

from mcp import Client, StdioServerParameters

SERVER = Path("dt4_server.py").resolve()
params = StdioServerParameters(command=sys.executable, args=[str(SERVER)])


async def ask(tool, **args):
    """Launch DT4, call one tool and return its structured result."""
    async with Client(params) as client:
        r = await client.call_tool(tool, args)
        if r.is_error:
            raise RuntimeError(r.content[0].text)
        return r.structured_content

## Discover the tools

In [2]:
async with Client(params) as client:
    for t in (await client.list_tools()).tools:
        print(f"{t.name:24s} {t.description[:88]}")

list_species             List the species, management regimes, disturbance types and growth models DT4 knows, wit
list_landscapes          List the pilot landscapes with their woodland area, CCI v7 years, climate and the CCI ca
get_agb                  Observed above-ground biomass (Mg/ha) of the woodland in a landscape, or part of it, in 
agb_timeseries           Annual CCI v7 AGB of a landscape or area for every year with a map (2005-2012, 2015-2024
carbon_stock             Living biomass carbon (above- and below-ground) of a landscape or area in a CCI year, pe
biomass_change           AGB change between two CCI years for a landscape or area: mean change with propagated un
attribute_change         Attribute the significant AGB loss between two CCI years to pressures, using evidence yo
simulate_stand           Simulate one stand forward with the yield-curve model, 3-PG, or both side by side: AGB, 
management_scenario      Compare management regimes for a landscape's typical stand (cal

## The pilot landscapes and their CCI v7 calibration

In [3]:
ls = await ask("list_landscapes")
for x in ls["landscapes"]:
    print(f"{x['name']}: {x['woodland_area_ha']:,.0f} ha woodland, {x['dominant_species']}, "
          f"{x['annual_rain_mm']:.0f} mm/yr, {x['mean_temperature_c']:.1f} C")
    for c in x["calibration"]:
        h = c["holdout_2018_2024"]
        print(f"   {c['growth_model']:12s} {c['productivity_name']:17s} median {c['productivity_median']:5.2f} "
              f"(IQR {c['productivity_p25']}-{c['productivity_p75']}), stand age {c['stand_age_median_2024']:.0f}, "
              f"hold-out RMSE {h['rmse_mg_ha']:.1f} Mg/ha, bias {h['bias_mg_ha']:+.1f}")

Thetford Forest: 14,198 ha woodland, CP, 604 mm/yr, 10.3 C
   yield_curve  yield_class       median  4.00 (IQR 3.0-8.0), stand age 58, hold-out RMSE 38.4 Mg/ha, bias +13.7
   3pg          fertility_rating  median  0.10 (IQR 0.05-0.35), stand age 62, hold-out RMSE 38.0 Mg/ha, bias +14.6
Kielder Forest: 18,979 ha woodland, SS, 1240 mm/yr, 7.6 C
   yield_curve  yield_class       median  6.00 (IQR 3.0-9.0), stand age 50, hold-out RMSE 42.6 Mg/ha, bias +0.9
   3pg          fertility_rating  median  0.20 (IQR 0.05-0.35), stand age 53, hold-out RMSE 43.7 Mg/ha, bias +1.8


## State: observed AGB and living carbon (EBCF role: state)

In [4]:
for key in ("thetford", "kielder"):
    a = await ask("get_agb", landscape=key, year=2024)
    c = await ask("carbon_stock", landscape=key, year=2024)
    print(f"{key:9s} 2024 AGB {a['agb_mean_mg_ha']:.1f} Mg/ha (90% {a['agb_p05_mg_ha']:.1f}-{a['agb_p95_mg_ha']:.1f}), "
          f"living C {c['living_c_t_ha']:.1f} t C/ha, total {c['total_living_co2e_t']/1e6:.2f} Mt CO2e")

ts = await ask("agb_timeseries", landscape="kielder")
print("\nKielder CCI v7 series (Mg/ha):", " ".join(f"{r['year']}:{r['agb_mean_mg_ha']:.0f}" for r in ts["series"]))
print(f"trend 2015-2024: {ts['trend_mg_ha_yr_2015_2024']:+.2f} Mg/ha/yr;", ts["sensor_epoch_note"])

pt = await ask("get_agb", landscape="kielder", year=2024, area={"point": [-2.55, 55.2], "radius_m": 500})
print(f"\nOne stand: {pt['area']}: {pt['agb_mean_mg_ha']:.1f} +- {pt['agb_sd_mg_ha']:.1f} Mg/ha over {pt['n_pixels']} pixels")

thetford  2024 AGB 106.4 Mg/ha (90% 85.3-127.6), living C 68.5 t C/ha, total 3.57 Mt CO2e


kielder   2024 AGB 116.8 Mg/ha (90% 87.3-146.3), living C 75.1 t C/ha, total 5.23 Mt CO2e



Kielder CCI v7 series (Mg/ha): 2005:91 2006:91 2007:91 2008:91 2009:92 2010:91 2011:92 2012:91 2015:103 2016:105 2017:107 2018:111 2019:112 2020:116 2021:118 2022:120 2023:119 2024:117
trend 2015-2024: +1.90 Mg/ha/yr; 2005-2012 maps use ALOS PALSAR/Envisat, 2015-2024 ALOS-2/Sentinel-1; compare levels across the gap with care.



One stand: point lon -2.5500, lat 55.2000, radius 500 m: 88.8 +- 32.2 Mg/ha over 86 pixels


## Impact and pressure: what changed, and why (EBCF roles: impact, pressure)

In [5]:
ch = await ask("biomass_change", landscape="kielder", year1=2015, year2=2024)
print(f"Kielder 2015-2024: {ch['delta_agb_mean_mg_ha']:+.1f} +- {ch['delta_agb_sd_mg_ha']:.1f} Mg/ha -> '{ch['impact']}'")
print(f"  significant loss on {ch['loss_area_ha']:,.0f} ha ({ch['conversion_candidate_area_ha']:,.0f} ha to below the woodland threshold), "
      f"gain on {ch['gain_area_ha']:,.0f} ha")

# attribution needs independent evidence; here a made-up felling footprint over the western half
at = await ask("attribute_change", landscape="kielder", year1=2015, year2=2024,
               evidence=[{"pressure": "harvest", "year_from": 2015, "year_to": 2024,
                          "bbox": [-2.70, 55.13, -2.525, 55.30], "source": "illustrative felling layer"}])
for r in at["attributions"]:
    print(f"  {r['label']}: {r['attributed_loss_area_ha']:,.0f} ha ({r['share_of_loss']:.0%})")
print(f"  unattributed: {at['unattributed_share']:.0%}  ({at['rule']})")

Kielder 2015-2024: +14.0 +- 25.3 Mg/ha -> 'no detectable change'
  significant loss on 467 ha (264 ha to below the woodland threshold), gain on 488 ha


  woody vegetation amount decrease (harvest): 359 ha (77%)
  unattributed: 23%  (A pressure is named only where independent evidence overlaps a significant loss in space and time (Living Earth EBCF).)


## Response: one stand, both growth models side by side

In [6]:
s = await ask("simulate_stand", landscape="kielder", age=20, years=50, regime="clearfell_no_thin")
for r in s["runs"]:
    agb = r["agb_mg_ha"]["mean"]
    print(f"{r['growth_model']:12s} {r['productivity_name']} {r['productivity']:.2f}: AGB "
          + " ".join(f"{agb[i]:.0f}" for i in range(0, 51, 10))
          + f" (every 10 yrs); cumulative net {r['cumulative_net_t_co2_ha'][0]:.0f} t CO2/ha "
          f"[{r['cumulative_net_t_co2_ha'][1]:.0f}, {r['cumulative_net_t_co2_ha'][2]:.0f}], "
          f"to soil (DT7) {r['litter_and_deadwood_to_soil_t_c_ha_yr']:.2f} t C/ha/yr")

yield_curve  yield_class 6.00: AGB 36 75 113 1 20 55 (every 10 yrs); cumulative net -237 t CO2/ha [-437, -72], to soil (DT7) 0.26 t C/ha/yr
3pg          fertility_rating 0.20: AGB 42 78 115 3 25 60 (every 10 yrs); cumulative net -203 t CO2/ha [-391, -42], to soil (DT7) 0.16 t C/ha/yr


## Response: management regimes for a typical stand

In [7]:
for key in ("kielder", "thetford"):
    m = await ask("management_scenario", landscape=key, horizon_years=60)
    print(f"{key}: start age {m['start_age']:.0f}, start AGB {m['start_agb_mg_ha']:.0f} Mg/ha; best for carbon {m['best_for_carbon']}")
    for o in m["outcomes"]:
        c = o["cumulative_net_t_co2_ha"]
        print(f"   {o['growth_model']:12s} {o['regime']:18s} net {c[0]:7.0f} t CO2/ha [{c[1]:6.0f},{c[2]:6.0f}]  "
              f"logs {o['removals_total_t_dm_ha']:5.0f} t/ha  final C {o['final_total_c_t_ha']:5.0f} t/ha")

kielder: start age 50, start AGB 117 Mg/ha; best for carbon {'yield_curve': 'continuous_cover', '3pg': 'no_intervention'}
   yield_curve  no_intervention    net    -315 t CO2/ha [  -448,  -187]  logs     0 t/ha  final C   137 t/ha
   yield_curve  thin_and_fell      net      30 t CO2/ha [  -153,   173]  logs   193 t/ha  final C    59 t/ha
   yield_curve  clearfell_no_thin  net     -53 t CO2/ha [  -290,   147]  logs   213 t/ha  final C    79 t/ha
   yield_curve  continuous_cover   net    -366 t CO2/ha [  -801,    13]  logs   141 t/ha  final C   152 t/ha
   3pg          no_intervention    net    -194 t CO2/ha [  -438,   -36]  logs     0 t/ha  final C   131 t/ha
   3pg          thin_and_fell      net      11 t CO2/ha [  -216,   198]  logs   210 t/ha  final C    67 t/ha
   3pg          clearfell_no_thin  net     -40 t CO2/ha [  -297,   180]  logs   215 t/ha  final C    81 t/ha
   3pg          continuous_cover   net    -157 t CO2/ha [  -516,    96]  logs   131 t/ha  final C   117 t/ha


thetford: start age 58, start AGB 106 Mg/ha; best for carbon {'yield_curve': 'continuous_cover', '3pg': 'no_intervention'}
   yield_curve  no_intervention    net    -263 t CO2/ha [  -369,  -176]  logs     0 t/ha  final C   120 t/ha
   yield_curve  thin_and_fell      net       8 t CO2/ha [  -170,   142]  logs   160 t/ha  final C    60 t/ha
   yield_curve  clearfell_no_thin  net    -215 t CO2/ha [  -582,    50]  logs    94 t/ha  final C   117 t/ha
   yield_curve  continuous_cover   net    -312 t CO2/ha [  -729,   -21]  logs   125 t/ha  final C   134 t/ha
   3pg          no_intervention    net    -220 t CO2/ha [  -518,   -55]  logs     0 t/ha  final C   132 t/ha
   3pg          thin_and_fell      net      13 t CO2/ha [  -215,   176]  logs   162 t/ha  final C    62 t/ha
   3pg          clearfell_no_thin  net    -114 t CO2/ha [  -443,   103]  logs    93 t/ha  final C    98 t/ha
   3pg          continuous_cover   net    -131 t CO2/ha [  -465,    78]  logs   116 t/ha  final C   104 t/ha


## Response: new woodland (the hook DT10 calls)

In [8]:
for sp in ("SS", "OK"):
    a = await ask("afforestation_scenario", landscape="kielder" if sp == "SS" else "thetford",
                  species=sp, area_ha=100, horizon_years=50)
    print(sp, {m: f"{v:.1f} t CO2/ha/yr" for m, v in a["carbon_per_ha_per_year_t_co2"].items()},
          {m: f"{v[0]/1000:.1f} kt CO2 on 100 ha" for m, v in a["total_cumulative_net_t_co2"].items()})

SS {'yield_curve': '-8.3 t CO2/ha/yr', '3pg': '-7.5 t CO2/ha/yr'} {'yield_curve': '-41.6 kt CO2 on 100 ha', '3pg': '-37.3 kt CO2 on 100 ha'}


OK {'yield_curve': '-8.1 t CO2/ha/yr', '3pg': '-7.2 t CO2/ha/yr'} {'yield_curve': '-40.5 kt CO2 on 100 ha', '3pg': '-35.9 kt CO2 on 100 ha'}


## Which growth model predicts CCI better? Hold-out validation

In [9]:
v = await ask("validate")
for r in v["rows"]:
    h = r["holdout"]
    print(f"{r['landscape']:9s} {r['growth_model']:12s} RMSE {h['rmse_mg_ha']:.1f}  bias {h['bias_mg_ha']:+.1f}  R2 {h['r2']:.2f}")
print("better:", v["better_model"])

thetford  yield_curve  RMSE 38.4  bias +13.7  R2 0.54
thetford  3pg          RMSE 38.0  bias +14.6  R2 0.55
kielder   yield_curve  RMSE 42.6  bias +0.9  R2 0.27
kielder   3pg          RMSE 43.7  bias +1.8  R2 0.24
better: {'thetford': '3pg', 'kielder': 'yield_curve'}


## Learning a site's productivity with the Kalman filter

In [10]:
a = await ask("assimilate_agb", landscape="kielder", site="demo-stand",
              area={"point": [-2.55, 55.2], "radius_m": 300}, persist=False)
for r in a["results"]:
    print(f"{r['growth_model']:12s} prior {r['prior_productivity']:.2f} +- {r['prior_productivity_sd']:.2f} -> "
          f"posterior {r['posterior_productivity']:.2f} +- {r['posterior_productivity_sd']:.2f}  ({r['data']})")

for m in ("yield_curve", "3pg"):
    s = await ask("self_learn_synthetic", growth_model=m)
    print(f"synthetic {m:12s} truth {s['true_productivity']:.2f}, prior {s['prior_productivity']:.2f} -> "
          f"recovered {s['recovered_productivity']:.2f} +- {s['recovered_sd']:.2f}")

yield_curve  prior 6.00 +- 4.45 -> posterior 4.44 +- 1.24  (CCI v7 2005-2024, point lon -2.5500, lat 55.2000, radius 300 m)
3pg          prior 0.20 +- 0.22 -> posterior 0.15 +- 0.06  (CCI v7 2005-2024, point lon -2.5500, lat 55.2000, radius 300 m)


synthetic yield_curve  truth 20.20, prior 9.80 -> recovered 19.06 +- 2.00


synthetic 3pg          truth 0.70, prior 0.30 -> recovered 0.68 +- 0.07
